In [3]:
import tkinter as tk
import random
import time
import sys
sys.setrecursionlimit(200000)

GOAL  = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE  = 90
GAP   = 5
ANIM  = 500
ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]
ANAMES  = {(-1,0):"UP",(1,0):"DOWN",(0,-1):"LEFT",(0,1):"RIGHT"}
MAX_DEPTH = 31

def apply_move(board, action):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    dr, dc = action
    nr, nc = r+dr, c+dc
    if 0 <= nr < 3 and 0 <= nc < 3:
        ni = nr*3+nc
        t  = list(board); t[idx], t[ni] = t[ni], t[idx]
        return tuple(t)
    return None

def misplaced(b):
    return sum(1 for i,v in enumerate(b) if v!=0 and v!=GOAL[i])

def is_solvable(b):
    arr = [x for x in b if x!=0]
    inv = sum(1 for i in range(len(arr))
              for j in range(i+1,len(arr)) if arr[i]>arr[j])
    return inv%2==0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t!=GOAL:
            return t

def fc_csp_search(initial, max_depth=MAX_DEPTH):
    nodes    = [0]
    found    = [False]
    sol_path = [None]
    log      = []

    def forward_checking(next_state, new_visited):
        return [d for d in ACTIONS
                if apply_move(next_state, d) is not None
                and apply_move(next_state, d) not in new_visited]

    def bt(depth, cur_state, path, visited):
        if found[0] or depth > max_depth:
            return
        nodes[0] += 1
        domain_cur = list(ACTIONS)  

        for action in ACTIONS:
            if found[0]: return
            aname      = ANAMES[action]
            next_state = apply_move(cur_state, action)

            if next_state is None:
                log.append({
                    "event"         : "INVALID",
                    "depth"         : depth,
                    "state_before"  : cur_state,
                    "action"        : action,
                    "action_name"   : aname,
                    "state_after"   : None,
                    "domain_current": domain_cur,
                    "domain_next"   : None,
                    "path_so_far"   : [ANAMES[a] for a in path]+[aname],
                    "reason"        : "out of bounds → invalid value",
                })
                continue

            if next_state in visited:
                log.append({
                    "event"         : "CYCLE",
                    "depth"         : depth,
                    "state_before"  : cur_state,
                    "action"        : action,
                    "action_name"   : aname,
                    "state_after"   : next_state,
                    "domain_current": domain_cur,
                    "domain_next"   : None,
                    "path_so_far"   : [ANAMES[a] for a in path]+[aname],
                    "reason"        : "next_state ∈ visited → cycle (no-repeat constraint violated)",
                })
                continue

            if next_state == GOAL:
                log.append({
                    "event"         : "GOAL",
                    "depth"         : depth,
                    "state_before"  : cur_state,
                    "action"        : action,
                    "action_name"   : aname,
                    "state_after"   : next_state,
                    "domain_current": domain_cur,
                    "domain_next"   : [],
                    "path_so_far"   : [ANAMES[a] for a in path]+[aname],
                    "reason"        : "GOAL reached ✓",
                })
                found[0]    = True
                sol_path[0] = path+[action]
                return

            new_visited = visited | {next_state}
            domain_next = forward_checking(next_state, new_visited)
            fc_prune    = (len(domain_next) == 0)

            if fc_prune:
                log.append({
                    "event"         : "FC_PRUNE",
                    "depth"         : depth,
                    "state_before"  : cur_state,
                    "action"        : action,
                    "action_name"   : aname,
                    "state_after"   : next_state,
                    "domain_current": domain_cur,
                    "domain_next"   : [],
                    "path_so_far"   : [ANAMES[a] for a in path]+[aname],
                    "reason"        : f"FC: domain(Move_{depth+2}) = ∅ → prune",
                })
                continue

            log.append({
                "event"         : "OK",
                "depth"         : depth,
                "state_before"  : cur_state,
                "action"        : action,
                "action_name"   : aname,
                "state_after"   : next_state,
                "domain_current": domain_cur,
                "domain_next"   : domain_next,
                "path_so_far"   : [ANAMES[a] for a in path]+[aname],
                "reason"        : f"FC: domain(Move_{depth+2}) = {[ANAMES[d] for d in domain_next]}",
            })
            bt(depth+1, next_state, path+[action], new_visited)
            if found[0]: return

    bt(0, initial, [], {initial})
    return nodes[0], sol_path[0], log
EVENT_STYLE = {
    "INVALID"  : ("#eeeeee", "#aaaaaa"),
    "CYCLE"    : ("#fff3e0", "#e65100"),  
    "FC_PRUNE" : ("#ffcccc", "#cc0000"),
    "OK"       : ("#c8e6c9", "#1b5e20"),
    "GOAL"     : ("#d0eeff", "#0055cc"),
}

class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle — CSP + Forward Checking (AIMA-correct)")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board   = random_board()
        self.log     = []
        self.log_idx = 0
        self.playing = False
        self.solution= None

        main = tk.Frame(root, bg="white"); main.pack(padx=10, pady=10)
        left = tk.Frame(main, bg="white"); left.pack(side="left", padx=10)

        self.canvas = tk.Canvas(left,
            width=3*TILE+4*GAP, height=3*TILE+4*GAP,
            bg="white", highlightthickness=0)
        self.canvas.pack()

        self.lbl_step = tk.Label(left, text="", font=("Arial",12), bg="white")
        self.lbl_step.pack(pady=4)

        self.lbl_h = tk.Label(left, text="h(n) = —", bg="white",
                              fg="#c05000", font=("Arial",10))
        self.lbl_h.pack()

        nf = tk.Frame(left, bg="white", bd=1, relief="groove")
        nf.pack(pady=4, fill="x", padx=4)
        tk.Label(nf, text="Event:", bg="white",
                 font=("Arial",9,"bold")).pack(side="left", padx=4)
        self.lbl_ntype = tk.Label(nf, text="Not started", bg="white",
                                  fg="#0055cc", font=("Consolas",10))
        self.lbl_ntype.pack(side="left", padx=4)

        r1 = tk.Frame(left, bg="white"); r1.pack(pady=2)
        self.lbl_nodes = tk.Label(r1, text="Nodes: —",   bg="white")
        self.lbl_nodes.grid(row=0, column=0, padx=5)
        self.lbl_steps = tk.Label(r1, text="Moves: —",   bg="white")
        self.lbl_steps.grid(row=0, column=1, padx=5)
        self.lbl_time  = tk.Label(r1, text="Time: — ms", bg="white")
        self.lbl_time.grid(row=0, column=2, padx=5)

        r2 = tk.Frame(left, bg="white"); r2.pack(pady=2)
        self.lbl_depth = tk.Label(r2, text="Var: —", bg="white",
                                  fg="#006400", font=("Arial",10))
        self.lbl_depth.grid(row=0, column=0, padx=5)
        self.lbl_path  = tk.Label(r2, text="Path: []", bg="white",
                                  fg="#00008b", font=("Arial",9))
        self.lbl_path.grid(row=0, column=1, padx=5)

        btns = tk.Frame(left, bg="white"); btns.pack(pady=6)
        tk.Button(btns, text="Shuffle", width=10,
                  command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10,
                  command=self.reset).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10,
                  command=self.solve).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step ▶",  width=34,
                  command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4)

        right = tk.Frame(main, bg="white"); right.pack(side="right", padx=10)
        tk.Label(right, text="Solution (Forward Checking)",
                 font=("Arial",13,"bold"), bg="white").pack()

        fc_box = tk.Frame(right, bg="#f0f8ff", bd=1, relief="groove")
        fc_box.pack(fill="x", padx=4, pady=4)
        tk.Label(fc_box,
                 text="Basic FC (CSP · Variables = Moves · Pure AIMA):",
                 bg="#f0f8ff", font=("Arial",9,"bold")).pack(anchor="w", padx=4)

        self.lbl_fc_var    = tk.Label(fc_box, text="Var      : —",
            bg="#f0f8ff", fg="#333", font=("Consolas",8))
        self.lbl_fc_var.pack(anchor="w", padx=8)

        self.lbl_fc_assign = tk.Label(fc_box, text="Try      : —",
            bg="#f0f8ff", fg="#333", font=("Consolas",8))
        self.lbl_fc_assign.pack(anchor="w", padx=8)

        self.lbl_fc_consist= tk.Label(fc_box, text="① Consist: —",
            bg="#f0f8ff", fg="#333", font=("Consolas",8))
        self.lbl_fc_consist.pack(anchor="w", padx=8)

        self.lbl_fc_domain = tk.Label(fc_box, text="② FC D+1 : —",
            bg="#f0f8ff", fg="#333", font=("Consolas",8))
        self.lbl_fc_domain.pack(anchor="w", padx=8)

        self.lbl_fc_result = tk.Label(fc_box, text="③ Result : —",
            bg="#f0f8ff", fg="#333", font=("Consolas",8,"bold"))
        self.lbl_fc_result.pack(anchor="w", padx=8, pady=(0,4))

        tk.Label(right, text="Domain[Move_i] = {UP, DOWN, LEFT, RIGHT}:",
                 bg="white", font=("Arial",9,"bold")).pack(anchor="w", padx=4)
        dgrid = tk.Frame(right, bg="white"); dgrid.pack(padx=4, pady=2)
        self.move_frames = {}
        for i, a in enumerate(ACTIONS):
            name = ANAMES[a]
            f   = tk.Frame(dgrid, bg="#f0f0f0", bd=1, relief="groove")
            f.grid(row=i//2, column=i%2, padx=4, pady=3)
            lbl = tk.Label(f, text=name, bg="#f0f0f0",
                           font=("Consolas",9,"bold"), width=14, height=2)
            lbl.pack()
            self.move_frames[name] = (f, lbl)

        self.sol = tk.Text(right, width=32, height=10, font=("Consolas",8))
        self.sol.pack(padx=4, pady=4)
        self.sol.tag_config("ok",     foreground="#006400")
        self.sol.tag_config("prune",  foreground="#cc0000")
        self.sol.tag_config("cycle",  foreground="#e65100")
        self.sol.tag_config("invalid",foreground="#aaaaaa")
        self.sol.tag_config("goal",   foreground="#0055cc",
                            font=("Consolas",8,"bold"))

        self._draw_board(self.board)
        self._reset_domain_grid()

    def _draw_board(self, board, moved_from=None):
        self.canvas.delete("all")
        self.lbl_h.config(text=f"h(n) = {misplaced(board)}")
        for i in range(9):
            r, c = divmod(i, 3)
            x1 = GAP+c*(TILE+GAP); y1 = GAP+r*(TILE+GAP)
            x2 = x1+TILE;          y2 = y1+TILE
            v  = board[i]
            if i == moved_from:
                fill, outline, dash = "#d0eeff", "#0055cc", ()
            elif v == 0:
                fill, outline, dash = "#e8f5e9", "#aaa",    (4,3)
            elif v == GOAL[i]:
                fill, outline, dash = "#f5f5f5", "#999",    ()
            else:
                fill, outline, dash = "#ffe0e0", "#cc0000", ()
            self.canvas.create_rectangle(x1,y1,x2,y2,
                fill=fill, outline=outline, width=2,
                dash=dash if dash else ())
            if v != 0:
                self.canvas.create_text((x1+x2)//2,(y1+y2)//2,
                    text=str(v), font=("Arial",28),
                    fill="#0055cc" if i==moved_from
                         else ("red" if v!=GOAL[i] else "black"))

    def _reset_domain_grid(self):
        for name, (f, lbl) in self.move_frames.items():
            lbl.config(text=f"{name}\n—", bg="#f0f0f0", fg="#555")
            f.config(bg="#f0f0f0")

    def _update_domain_grid(self, action, event, domain_next):
        chosen     = ANAMES[action] if action else None
        next_names = {ANAMES[a] for a in (domain_next or [])}

        for name, (f, lbl) in self.move_frames.items():
            if name == chosen:
                bg, fg = EVENT_STYLE[event]
                labels = {
                    "INVALID"  : "✗ out of bounds",
                    "CYCLE"    : "⊘ cycle → SKIP",
                    "FC_PRUNE" : "✗ FC PRUNE",
                    "OK"       : "✓ assign",
                    "GOAL"     : "★ GOAL",
                }
                note = labels[event]
            elif event in ("OK", "FC_PRUNE", "GOAL") and domain_next is not None:
                if name in next_names:
                    bg, fg, note = "#f5f5f5", "#333", "✓ in D+1"
                else:
                    bg, fg, note = "#fff3e0", "#e65100", "⊘ FC removed from D+1"
            else:
                bg, fg, note = "#f0f0f0", "#555", "in current D"

            lbl.config(text=f"{name}\n{note}", bg=bg, fg=fg)
            f.config(bg=bg)

    def _reset(self):
        self.log=[]; self.log_idx=0; self.playing=False; self.solution=None
        for w, t in [(self.lbl_nodes,"Nodes: —"),(self.lbl_steps,"Moves: —"),
                     (self.lbl_time,"Time: — ms"),(self.lbl_depth,"Var: —"),
                     (self.lbl_path,"Path: []"),(self.lbl_step,""),
                     (self.lbl_h,"h(n) = —")]:
            w.config(text=t)
        self.lbl_ntype.config(text="Not started", fg="#0055cc")
        for w, t in [(self.lbl_fc_var,"Var      : —"),
                     (self.lbl_fc_assign,"Try      : —"),
                     (self.lbl_fc_consist,"① Consist: —"),
                     (self.lbl_fc_domain,"② FC D+1 : —"),
                     (self.lbl_fc_result,"③ Result : —")]:
            w.config(text=t, fg="#333")
        self.sol.delete("1.0", tk.END)
        self._reset_domain_grid()

    def shuffle(self):
        self.board = random_board(); self._reset()
        self._draw_board(self.board)

    def reset(self):
        self.board = GOAL; self._reset()
        self._draw_board(self.board)

    def solve(self):
        if self.playing: return
        self._reset()
        self._draw_board(self.board)

        t0 = time.perf_counter()
        nodes, sol_path, log = fc_csp_search(self.board)
        ms = (time.perf_counter()-t0)*1000

        self.log      = log
        self.log_idx  = 0
        self.solution = sol_path

        solved     = (sol_path is not None)
        move_count = len(sol_path) if sol_path else 0

        self.lbl_nodes.config(text=f"Nodes: {nodes}")
        self.lbl_steps.config(text=f"Moves: {move_count}")
        self.lbl_time .config(text=f"Time: {ms:.1f} ms")
        self.lbl_step .config(text="✓ SOLVED" if solved else "✗ Failed")

        if log:
            self.playing = True
            self.animate()

    def _show_entry(self, idx):
        if idx >= len(self.log): return
        e = self.log[idx]

        event   = e["event"]
        depth   = e["depth"]
        action  = e["action"]
        aname   = e["action_name"]
        sb      = e["state_before"]
        sa      = e["state_after"]
        d_next  = e["domain_next"]
        path_sf = e["path_so_far"]

        blank_from = sb.index(0)
        board_show = sa if (sa is not None and event not in ("INVALID","CYCLE")) else sb
        self._draw_board(board_show,
                         moved_from=blank_from if event not in ("INVALID","CYCLE") else None)
        self._update_domain_grid(action, event, d_next)

        self.lbl_depth.config(text=f"Var: Move_{depth+1}")
        path_short = path_sf[-5:]
        self.lbl_path.config(
            text="Path: [" + ", ".join(path_short) +
                 ("…" if len(path_sf)>5 else "") + "]")

        self.lbl_fc_var.config(text=f"Var      : Move_{depth+1}")
        self.lbl_fc_assign.config(text=f"Try      : Move_{depth+1} = {aname}")

        if event == "INVALID":
            self.lbl_fc_consist.config(
                text="① Consist: ✗  out of bounds", fg="#aaaaaa")
            self.lbl_fc_domain.config(
                text="② FC D+1 : (skipped — invalid value)", fg="#aaaaaa")
            self.lbl_fc_result.config(
                text="③ Result : SKIP — invalid value", fg="#aaaaaa")
            self.lbl_step.config(text=f"Move_{depth+1}={aname} → INVALID ✗")
            self.lbl_ntype.config(text="Invalid", fg="#aaaaaa")

        elif event == "CYCLE":
            self.lbl_fc_consist.config(
                text="① Consist: ✗  next ∈ visited", fg="#e65100")
            self.lbl_fc_domain.config(
                text="② FC D+1 : (skipped — no-repeat constraint violated)", fg="#aaaaaa")
            self.lbl_fc_result.config(
                text="③ Result : SKIP — cycle detected", fg="#e65100")
            self.lbl_step.config(text=f"Move_{depth+1}={aname} → CYCLE ⊘")
            self.lbl_ntype.config(text="Cycle (no-repeat)", fg="#e65100")

        elif event == "FC_PRUNE":
            self.lbl_fc_consist.config(
                text="① Consist: ✓  valid, not visited", fg="#006400")
            self.lbl_fc_domain.config(
                text="② FC D+1 : ∅  → PRUNE", fg="#cc0000")
            self.lbl_fc_result.config(
                text="③ Result : FC PRUNE → backtrack", fg="#cc0000")
            self.lbl_step.config(text=f"Move_{depth+1}={aname} → FC PRUNE ✗")
            self.lbl_ntype.config(text="FC Prune!", fg="#cc0000")

        elif event == "GOAL":
            self.lbl_fc_consist.config(
                text="① Consist: ✓  valid, not visited", fg="#006400")
            self.lbl_fc_domain.config(
                text="② FC D+1 : GOAL → FC not needed", fg="#0055cc")
            self.lbl_fc_result.config(
                text="③ Result : GOAL ✓", fg="#0055cc")
            self.lbl_step.config(text=f"Move_{depth+1}={aname} → GOAL ✓")
            self.lbl_ntype.config(text="GOAL ✓", fg="#0055cc")

        else: 
            next_names = [ANAMES[a] for a in d_next] if d_next else []
            self.lbl_fc_consist.config(
                text="① Consist: ✓  valid, not visited", fg="#006400")
            self.lbl_fc_domain.config(
                text=f"② FC D+1 : {{{', '.join(next_names)}}}", fg="#006400")
            self.lbl_fc_result.config(
                text=f"③ Result : OK → assign Move_{depth+1} + recurse",
                fg="#006400")
            self.lbl_step.config(text=f"Move_{depth+1}={aname} → OK ✓")
            self.lbl_ntype.config(text="Assign var", fg="#cc5500")

        self.sol.delete("1.0", tk.END)
        for i, en in enumerate(self.log[:idx+1]):
            p, an, ev = en["depth"], en["action_name"], en["event"]
            line = f"{i+1:3}. Move_{p+1}={an:<5}  "
            if ev == "GOAL":
                self.sol.insert(tk.END, line+"GOAL ✓\n",       "goal")
            elif ev == "FC_PRUNE":
                self.sol.insert(tk.END, line+"FC_PRUNE ✗\n",   "prune")
            elif ev == "CYCLE":
                self.sol.insert(tk.END, line+"CYCLE ⊘\n",      "cycle")
            elif ev == "INVALID":
                self.sol.insert(tk.END, line+"INVALID ✗\n",    "invalid")
            else:
                dn = [ANAMES[a] for a in en["domain_next"]] if en["domain_next"] else []
                self.sol.insert(tk.END, line+f"ok d+1={dn}\n", "ok")
        self.sol.see(tk.END)

    def animate(self):
        if not self.playing: return
        self._show_entry(self.log_idx)
        if self.log_idx < len(self.log)-1:
            self.log_idx += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False
            if self.solution:
                self._draw_board(GOAL)
                self.lbl_step .config(text="✓ SOLVED!")
                self.lbl_ntype.config(text="Done", fg="#006400")
            else:
                self.lbl_step .config(text="✗ No solution found")
                self.lbl_ntype.config(text="Failed", fg="#cc0000")

    def next_step(self):
        self.playing = False
        if not self.log: return
        if self.log_idx < len(self.log)-1:
            self.log_idx += 1
        self._show_entry(self.log_idx)


if __name__ == "__main__":
    root = tk.Tk()
    App(root)
    root.mainloop()